# 6-bob. 6-BOB. FISHING VA SPAMNI SUN'IY INTELLEKT YORDAMIDA ANIQLASH
## Kengaytirilgan amaliy ish: o'xshash domenlarni aniqlovchi kuzatuv tizimi

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/06-bob/6-2_oxshash-domenlarni-aniqlovchi-kuzatuv-tizimi.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 6-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dastur himoyalanayotgan beshta domen uchun yuqoridagi uchta tekshiruvni amalga oshiradi. U qo'shni harflar o'rnini almashtirishni ham hisobga oluvchi Damerau–Levenshtein masofasidan va Python'ning standart "idna" kodlashidan foydalanadi. Sinov uchun yangi ro'yxatdan o'tgan domenlar oqimi modellashtirilgan.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import unicodedata

HIMOYALANGAN = ["namunabank.uz", "uzcard.uz", "payme.uz", "my.gov.uz", "tsue.uz"]

In [2]:
# 1. "Skelet": ko'rinishi o'xshash belgilarni bitta asosiy belgiga keltirish (Unicode UTS #39 g'oyasi)
OXSHASH = {"а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x", "і": "i",
           "ј": "j", "ԁ": "d", "ɡ": "g", "0": "o", "1": "l", "3": "e", "5": "s", "!": "i"}
def skelet(domen):
    d = unicodedata.normalize("NFKC", domen.lower())
    d = "".join(OXSHASH.get(b, b) for b in d)
    return d.replace("rn", "m").replace("vv", "w")

In [3]:
# 2. Damerau-Levenshtein masofasi: qo'shish, o'chirish, almashtirish va qo'shni harflar o'rnini almashtirish
def masofa(a, b):
    d = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1): d[i][0] = i
    for j in range(len(b) + 1): d[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            narx = 0 if a[i - 1] == b[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + narx)
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                d[i][j] = min(d[i][j], d[i - 2][j - 2] + 1)
    return d[-1][-1]

def tekshir(domen):
    s = skelet(domen)
    for brend in HIMOYALANGAN:
        if domen == brend:
            return "asl domen", brend
        if s == brend:
            return "o'xshash belgilar (gomoglif)", brend
        if masofa(s, brend) <= 2:
            return f"harf xatosi (masofa {masofa(s, brend)})", brend
        nom = brend.split(".")[0]
        if len(nom) >= 5 and nom in s.replace("-", "").replace(".", ""):
            return "brend nomi boshqa domen ichida", brend
    return "bog'liqlik topilmadi", "-"

In [4]:
# 3. Yangi ro'yxatdan o'tgan domenlar oqimi (masalan, sertifikatlar shaffofligi jurnallaridan)
oqim = ["namunabank.uz", "namunbank.uz", "namunabakn.uz", "nаmunаbаnk.uz", "uzcаrd.uz",
        "uzcard-bonus.top", "payme.uz-verify.com", "pауme.uz", "my-gov.uz", "rny.gov.uz",
        "tsue.uz", "tsue-edu.uz", "kun.uz", "olx.uz", "namunabank-uz.com", "paymebank.uz"]
print("{:<22}{:<34}{:<32}{}".format("Domen", "DNSdagi ko'rinishi (punycode)", "Xulosa", "Brend"))
for d in oqim:
    try:
        puny = d.encode("idna").decode()
    except UnicodeError:
        puny = "(kodlab bo'lmadi)"
    xulosa, brend = tekshir(d)
    print(f"{d:<22}{puny:<34}{xulosa:<32}{brend}")

Domen                 DNSdagi ko'rinishi (punycode)     Xulosa                          Brend
namunabank.uz         namunabank.uz                     asl domen                       namunabank.uz
namunbank.uz          namunbank.uz                      harf xatosi (masofa 1)          namunabank.uz
namunabakn.uz         namunabakn.uz                     harf xatosi (masofa 1)          namunabank.uz
nаmunаbаnk.uz         xn--nmunbnk-2fgdb.uz              o'xshash belgilar (gomoglif)    namunabank.uz
uzcаrd.uz             xn--uzcrd-6ve.uz                  o'xshash belgilar (gomoglif)    uzcard.uz
uzcard-bonus.top      uzcard-bonus.top                  brend nomi boshqa domen ichida  uzcard.uz
payme.uz-verify.com   payme.uz-verify.com               brend nomi boshqa domen ichida  payme.uz
pауme.uz              xn--pme-6cd9g.uz                  o'xshash belgilar (gomoglif)    payme.uz
my-gov.uz             my-gov.uz                         harf xatosi (masofa 1)          my.gov.uz
rny.gov.uz

## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Domen                 DNSdagi ko'rinishi (punycode)     Xulosa                          Brend
namunabank.uz         namunabank.uz                     asl domen                       namunabank.uz
namunbank.uz          namunbank.uz                      harf xatosi (masofa 1)          namunabank.uz
namunabakn.uz         namunabakn.uz                     harf xatosi (masofa 1)          namunabank.uz
nаmunаbаnk.uz         xn--nmunbnk-2fgdb.uz              o'xshash belgilar (gomoglif)    namunabank.uz
uzcаrd.uz             xn--uzcrd-6ve.uz                  o'xshash belgilar (gomoglif)    uzcard.uz
uzcard-bonus.top      uzcard-bonus.top                  brend nomi boshqa domen ichida  uzcard.uz
payme.uz-verify.com   payme.uz-verify.com               brend nomi boshqa domen ichida  payme.uz
pауme.uz              xn--pme-6cd9g.uz                  o'xshash belgilar (gomoglif)    payme.uz
my-gov.uz             my-gov.uz                         harf xatosi (masofa 1)          my.gov.uz
rny.gov.uz            rny.gov.uz                        o'xshash belgilar (gomoglif)    my.gov.uz
tsue.uz               tsue.uz                           asl domen                       tsue.uz
tsue-edu.uz           tsue-edu.uz                       bog'liqlik topilmadi            -
kun.uz                kun.uz                            bog'liqlik topilmadi            -
olx.uz                olx.uz                            bog'liqlik topilmadi            -
namunabank-uz.com     namunabank-uz.com                 brend nomi boshqa domen ichida  namunabank.uz
paymebank.uz          paymebank.uz                      brend nomi boshqa domen ichida  payme.uz
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 6-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.